# Collinearity

Two predictors that carry the same information make $X^{\mathsf T}X$ singular or nearly singular. Exact dependence means the coefficients are not identified. Near dependence means they are identified and unstable: a small change in $y$ moves them by a large amount.


## Exact dependence

Let $x_1 = 1, 2, 3, 4$ and $x_2 = 2x_1$. The third column of $X$ is twice the second, so

$$
\det(X^{\mathsf T}X) = 0.
$$

The laws $y = 3 + 2x_1 + 0\cdot x_2$ and $y = 3 + 0\cdot x_1 + 1\cdot x_2$ produce the same predictions on this table, because $x_2 = 2x_1$ turns both into $3 + 2x_1$. Infinitely many coefficient vectors share one fitted vector. No regression program can report "the" slope of $x_1$ from these columns.


In [1]:
# Twice the first predictor is not a second predictor.
import numpy as np
x1 = np.array([1.0, 2, 3, 4])
x2 = 2 * x1
X = np.column_stack([np.ones(4), x1, x2])
gram = X.T @ X
print("X'X")
print(gram)
print("determinant", np.linalg.det(gram))
assert abs(np.linalg.det(gram)) < 1e-8
try:
    np.linalg.solve(gram, np.array([1.0, 0, 0]))
    raise AssertionError("expected a singular Gram matrix")
except np.linalg.LinAlgError:
    print("exact collinearity: LinAlgError")


X'X
[[  4.  10.  20.]
 [ 10.  30.  60.]
 [ 20.  60. 120.]]
determinant 0.0
exact collinearity: LinAlgError


## A near dependence, and a small edit

Now let $x_2 = 2, 4, 6, 9$, so the last entry breaks the exact doubling. Define the response from a known law,

$$
y = 3 + 2x_1 - x_2,
$$

which gives $y = 3, 3, 3, 2$. The columns are full rank, and least squares recovers $(3, 2, -1)$ with residual sum of squares zero.

Add $1/10$ to the last response only: $y = 3, 3, 3, 21/10$. Solving the perturbed normal equations yields

$$
\widehat\beta = \left(3,\ \dfrac{9}{5},\ -\dfrac{9}{10}\right).
$$

The intercept did not move. The slopes moved by $1/5$ and $1/10$. One tenth of a unit in a single response was multiplied by the near dependence.

The same edit on a more separated second predictor, $x_2 = 1, 3, 2, 4$, starts from $y = 4, 4, 7, 7$ and becomes $4, 4, 7, 7.1$. The perturbed coefficients are

$$
\dfrac{353}{120},\ \dfrac{121}{60},\ -\dfrac{59}{60},
$$

which differ from $(3, 2, -1)$ by only $7/120$, $1/60$, and $1/60$. The same $1/10$ error moves the separated slopes by a sixtieth and the nearly dependent slopes by a fifth and a tenth.


In [2]:
# A tenth added to one response moves the nearly dependent slopes much farther.
from fractions import Fraction
import numpy as np
x1 = np.array([1.0, 2, 3, 4])
near = np.array([2.0, 4, 6, 9])
spread = np.array([1.0, 3, 2, 4])
y_near = np.array([3.0, 3, 3, 2.1])
y_spread = np.array([4.0, 4, 7, 7.1])
X_near = np.column_stack([np.ones(4), x1, near])
X_spread = np.column_stack([np.ones(4), x1, spread])
beta_near = np.linalg.solve(X_near.T @ X_near, X_near.T @ y_near)
beta_spread = np.linalg.solve(X_spread.T @ X_spread, X_spread.T @ y_spread)
print("near", beta_near)
print("spread", beta_spread)
print("condition numbers", np.linalg.cond(X_near.T @ X_near), np.linalg.cond(X_spread.T @ X_spread))
assert np.allclose(beta_near, [3, 9 / 5, -9 / 10])
assert np.allclose(beta_spread, [353 / 120, 121 / 60, -59 / 60])
assert abs(beta_near[1] - 2) > abs(beta_spread[1] - 2) * 5


near [ 3.   1.8 -0.9]
spread [ 2.94166667  2.01666667 -0.98333333]
condition numbers 3780.799981391993 108.24076133623076


## The variance inflation factor

Regress the nearly dependent $x_1$ on $x_2$ with an intercept. The centered sums are

$$
\begin{aligned}
S_{x_2 x_2} &= \dfrac{107}{4}, & S_{x_1 x_2} &= \dfrac{23}{2}, \\
b_1 &= \dfrac{23/2}{107/4} = \dfrac{46}{107}, &
b_0 &= \dfrac{5}{2} - \dfrac{46}{107}\cdot\dfrac{21}{4} = \dfrac{26}{107}.
\end{aligned}
$$

The residual sum of squares of this auxiliary regression is $642/11449$, and the total sum of squares of $x_1$ is $5$. Hence

$$
R^2_{1} = 1 - \dfrac{642/11449}{5} = \dfrac{529}{535}, \qquad
\mathrm{VIF}_1 = \dfrac{1}{1 - R^2_{1}} = \dfrac{535}{6}.
$$

A variance inflation factor of $535/6 \approx 89.2$ means the estimated variance of the coefficient of $x_1$, in a model that also contains this $x_2$, is about eighty-nine times the variance that same coefficient would have if $x_2$ were uncorrelated with $x_1$. The condition number of $X^{\mathsf T}X$ told the same story in one matrix. The variance inflation factor assigns the trouble to a column.


In [3]:
# VIF = 1 / (1 - R^2) from the regression of x1 on x2.
from fractions import Fraction
import numpy as np
x1 = np.array([1.0, 2, 3, 4])
x2 = np.array([2.0, 4, 6, 9])
X = np.column_stack([np.ones(4), x2])
beta = np.linalg.solve(X.T @ X, X.T @ x1)
residual = x1 - X @ beta
tss = np.sum((x1 - x1.mean()) ** 2)
r2 = 1 - np.sum(residual ** 2) / tss
vif = 1 / (1 - r2)
print("auxiliary beta", beta)
print("R2", r2)
print("VIF", vif)
assert np.allclose(beta, [26 / 107, 46 / 107])
assert abs(r2 - 529 / 535) < 1e-12
assert abs(vif - 535 / 6) < 1e-12


auxiliary beta [0.24299065 0.42990654]
R2 0.9887850467289719
VIF 89.16666666666649


## Pitfall

A large coefficient is not evidence against collinearity, and a small coefficient is not evidence for it. In the exact fit before the perturbation, the slopes were the clean numbers $2$ and $-1$ while the variance inflation factor was already $535/6$. Collinearity is a property of the columns of $X$. The response $y$ does not enter $X^{\mathsf T}X$, the condition number, or the variance inflation factor.


In [4]:
# The Gram matrix, and therefore the VIF, ignores y.
import numpy as np
x1 = np.array([1.0, 2, 3, 4])
x2 = np.array([2.0, 4, 6, 9])
X = np.column_stack([np.ones(4), x1, x2])
gram = X.T @ X
other_response = np.array([0.0, 1, 0, 1])
print("same X'X for every y", np.array_equal(gram, X.T @ X))
print("y is not a factor of X'X", other_response.tolist())
assert gram.shape == (3, 3)


same X'X for every y True
y is not a factor of X'X [0.0, 1.0, 0.0, 1.0]


## Summary

- Exact collinearity makes $X^{\mathsf T}X$ singular and the coefficients unidentified.
- Adding $1/10$ to one response moved the nearly dependent slopes by $1/5$ and $1/10$, and the separated slopes by only $1/60$.
- For $x_1$ on this $x_2$, $R^2 = 529/535$ and $\mathrm{VIF} = 535/6$.
